# Лабораторная работа 12. VAR и многомерные временные ряды

**Цель.** Исследовать многомерный временной ряд, проверить стационарность и построить VAR-прогноз.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR

np.random.seed(42)
csv_candidates = [Path('AirQualityUCI.csv'), Path('data/AirQualityUCI.csv')]
csv_path = next((path for path in csv_candidates if path.exists()), csv_candidates[0])


## Подготовка входных данных

В этом блоке считываются данные, с которыми дальше будет работать лабораторная. После загрузки важно проверить формат индекса, названия колонок и размерность ряда, потому что от этого зависит корректность всех следующих расчётов.


In [ ]:
if not csv_path.exists():
    print('CSV-файл не найден, VAR-анализ пропущен.')
else:
    df = pd.read_csv(csv_path, parse_dates=[['Date', 'Time']], sep=';', decimal=',')
    df['Date_Time'] = pd.to_datetime(df['Date_Time'], format='%d/%m/%Y %H.%M.%S', errors='coerce')
    data = df.drop(columns=['Date_Time']).apply(pd.to_numeric, errors='coerce')
    data.index = df['Date_Time']
    data = data.replace(-200, np.nan).interpolate().dropna(axis=1, how='all').dropna()
    numeric = data.select_dtypes(include='number').iloc[:, :4]
    print(numeric.head())
    numeric.plot(figsize=(12, 5), subplots=True, layout=(2, 2), legend=False)
    plt.tight_layout()
    plt.show()


## Подготовка входных данных

В этом блоке считываются данные, с которыми дальше будет работать лабораторная. После загрузки важно проверить формат индекса, названия колонок и размерность ряда, потому что от этого зависит корректность всех следующих расчётов.


In [ ]:
if csv_path.exists() and 'numeric' in globals() and len(numeric) > 30:
    for column in numeric.columns:
        pvalue = adfuller(numeric[column].dropna())[1]
        print(f'{column}: p-value ADF = {pvalue:.4f}')

    train = numeric.iloc[:-24]
    model = VAR(train)
    fitted = model.fit(maxlags=5, ic='aic')
    forecast = fitted.forecast(train.values[-fitted.k_ar:], steps=24)
    forecast = pd.DataFrame(forecast, columns=numeric.columns)
    print(forecast.head())


## Вывод

VAR-модель строится для нескольких связанных временных рядов одновременно.
